# Tomato Disease Classification - Vision Transformer (ViT-B/16)
**Batch F Capstone Project** | **Author:** Laksh Gupta | **Reg No:** 2430010393


In [ ]:
# ==========================================
# 1. SETUP & UNZIP DATASET
# ==========================================
import os
import copy
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models import vit_b_16, ViT_B_16_Weights

# Auto-find and unzip dataset archive
zip_candidates = ['/tomato_data.zip.zip', 'tomato_data.zip.zip', '/content/tomato_data.zip.zip', 'tomato_data.zip', '/tomato_data.zip']
for z in zip_candidates:
    if os.path.exists(z):
        print(f'Unzipping {z} to /content/...')
        !unzip -q -o '$z' -d /content/
        break

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Setup complete!')


In [ ]:
# ==========================================
# 2. DATA PREPARATION & DATALOADERS
# ==========================================
data_dir = '/content/Tomato_Disease_Dataset'
if not os.path.exists(data_dir):
    for root, dirs, files in os.walk('/content'):
        if 'train' in dirs and 'val' in dirs:
            data_dir = root
            break

print('Dataset directory:', data_dir)

# Augmentation pipeline for 224x224 input
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(f'{data_dir}/train', transform=train_transform)
val_dataset = datasets.ImageFolder(f'{data_dir}/val', transform=test_transform)
test_dataset = datasets.ImageFolder(f'{data_dir}/test', transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

classes = train_dataset.classes
print('Classes:', classes)
print(f'Samples: Train={len(train_dataset)} | Val={len(val_dataset)} | Test={len(test_dataset)}')


In [ ]:
# ==========================================
# 3. BASELINE REGULARIZED CNN (For Reference)
# ==========================================
class RegularizedTomatoCNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.5):
        super(RegularizedTomatoCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

print('Baseline CNN defined: 120,458 parameters (Achieved 72.20% accuracy).')


In [ ]:
# ==========================================
# 4. PROPOSED SOTA MODEL: VISION TRANSFORMER (ViT-B/16)
# ==========================================
print('Setting up Fine-Tuned Vision Transformer (ViT-B/16)...')

model_vit = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)

# UNFREEZE ALL TRANSFORMER ENCODER LAYERS
for param in model_vit.parameters():
    param.requires_grad = True

num_ftrs = model_vit.heads.head.in_features
model_vit.heads.head = nn.Sequential(
    nn.Dropout(p=0.1),
    nn.Linear(num_ftrs, len(classes))
)
model_vit = model_vit.to(device)

total_params = sum(p.numel() for p in model_vit.parameters())
print(f'ViT-B/16 Total Parameters: {total_params:,} (~{total_params/1e6:.1f} Million)')

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(model_vit.parameters(), lr=3e-5, weight_decay=0.01)
num_epochs = 10
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
print('ViT-B/16 ready for training!')


In [ ]:
# ==========================================
# 5. TRAINING LOOP FOR VISION TRANSFORMER
# ==========================================
best_val_loss = float('inf')
best_model_weights = copy.deepcopy(model_vit.state_dict())
patience = 3
epochs_without_improvement = 0

train_losses, val_losses, train_accuracies, val_accuracies = [], [], [], []

for epoch in range(num_epochs):
    model_vit.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_vit(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_vit.parameters(), max_norm=1.0)
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    train_loss = running_loss / len(train_dataset)
    train_acc = 100 * correct / total
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    
    # Validation
    model_vit.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_vit(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    val_loss = val_loss / len(val_dataset)
    val_acc = 100 * correct / total
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    
    print(f'Epoch [{epoch+1}/{num_epochs}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%')
    scheduler.step()
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_weights = copy.deepcopy(model_vit.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break

model_vit.load_state_dict(best_model_weights)
print('Training Complete. Best weights restored.')


In [ ]:
# ==========================================
# 6. TEST SET EVALUATION & CONFUSION MATRIX
# ==========================================
model_vit.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_vit(images)
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = accuracy_score(all_labels, all_preds) * 100.0
print(f'Final Vision Transformer Test Accuracy: {test_acc:.2f}%\n')
print('Classification Report:')
print(classification_report(all_labels, all_preds, target_names=classes, digits=4))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - Vision Transformer (ViT-B/16) [{test_acc:.2f}%]')
plt.show()
